# Robot arm example

In [1]:
#@title Install libraries
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vedo"])
print("vedo installed")

vedo installed


In [2]:
# Optional timing extension
try:
    import autotime
    %load_ext autotime
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install ipython-autotime
    %load_ext autotime

time: 103 μs (started: 2026-03-19 02:34:47 -04:00)


In [3]:
#@title Import libraries
import vedo

if "google.colab" in str(get_ipython()):
    vedo.settings.init_colab()


import os
import glob
import numpy as np
import vedo

# Force VTK backend (important in Colab)
if "google.colab" in str(get_ipython()):
    if vedo.settings.default_backend != "vtk":
        print("Switching VTK backend for compatibility.")
        vedo.settings.default_backend = "vtk"



time: 286 ms (started: 2026-03-19 02:34:47 -04:00)


In [4]:
#@title Re-download and extract robot files cleanly

import os
import shutil
import zipfile
import urllib.request

zip_url = "https://www.dropbox.com/scl/fi/uewvrcempf2wf2jp7bcb8/robot.zip?rlkey=7uwz1ne94hxyinub8x16y93em&dl=1"
zip_path = "robot.zip"
extract_dir = "robot"

# remove broken leftovers
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)

if os.path.exists(zip_path):
    os.remove(zip_path)

print("Downloading robot.zip ...")
urllib.request.urlretrieve(zip_url, zip_path)

print("Extracting robot.zip ...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(".")

print("Cleaning up zip ...")
os.remove(zip_path)

print("Done. Checking files:")
for root, dirs, files in os.walk("."):
    for f in files:
        if f.lower().endswith(".stl"):
            print(os.path.join(root, f))

Extracting robot.zip ...
Cleaning up zip ...
Done. Checking files:
.\robot\Base.stl
.\robot\BaseRot.stl
.\robot\Humerus.stl
.\robot\Radius.stl
time: 2.17 s (started: 2026-03-19 02:34:47 -04:00)


In [5]:
#@title Imports
import os
import glob
import numpy as np
import vedo

from vedo import (
    load,
    Axes,
    Circle,
    Arrow,
    Sphere,
    Plotter,
    LinearTransform,
)

time: 871 μs (started: 2026-03-19 02:34:49 -04:00)


In [6]:
#@title Robot class

class RobotArm:
    """
    Robot arm with:
      - forward kinematics
      - Jacobian-based IK
      - persistent vedo meshes updated in-place
      - single-step IK for reactive control in the crowd simulation
    """

    def __init__(self, partLengths, parts, arm_location):
        # Arm location (position of the first frame)
        self.arm_location = np.array(arm_location, dtype=float).reshape(3, 1)

        # Lengths
        self.L1, self.L2, self.L3, self.L4 = partLengths

        # Store source meshes (do not mutate these directly)
        self.source_parts = parts

        # Constants
        self.delta_phi = 0.1              # finite-difference step, in degrees
        self.target = np.array([0.0, 100.0, 200.0], dtype=float)
        self.target_tolerance = 30.0
        self.target_lambda = 0.001
        self.convergence = 0.02
        self.iteration_limit = 1000

        # Live meshes for rendering
        self.meshes = None
        self.initialize_meshes()

    def initialize_meshes(self):
        """
        Create robot meshes once for this robot instance.
        These are the only meshes used during animation.
        """
        Base  = self.source_parts[0].clone()
        Part1 = self.source_parts[1].clone()
        Part2 = self.source_parts[2].clone()
        Part3 = self.source_parts[3].clone()
        Part4 = self.createCoordinateFrameMesh()
        self.meshes = [Base, Part1, Part2, Part3, Part4]

    def RotationMatrix(self, theta, axis_name):
        """
        Single-axis rotation matrix.
        theta is interpreted in degrees.
        """
        c = np.cos(theta * np.pi / 180.0)
        s = np.sin(theta * np.pi / 180.0)

        if axis_name == "x":
            rotation_matrix = np.array([
                [1, 0,  0],
                [0, c, -s],
                [0, s,  c]
            ], dtype=float)
        elif axis_name == "y":
            rotation_matrix = np.array([
                [ c, 0, s],
                [ 0, 1, 0],
                [-s, 0, c]
            ], dtype=float)
        elif axis_name == "z":
            rotation_matrix = np.array([
                [c, -s, 0],
                [s,  c, 0],
                [0,  0, 1]
            ], dtype=float)
        else:
            raise ValueError(f"Unknown axis_name: {axis_name}")

        return rotation_matrix

    def createCoordinateFrameMesh(self):
        """
        Returns a mesh representing a coordinate frame.
        """
        shaft_radius = 0.05
        head_radius = 0.10
        alpha = 1.0
        unit = 30

        x_axisArrow = Arrow(
            start_pt=(0, 0, 0),
            end_pt=(unit, 0, 0),
            shaft_radius=shaft_radius,
            head_radius=head_radius,
            res=12,
            c="red",
            alpha=alpha,
        )

        y_axisArrow = Arrow(
            start_pt=(0, 0, 0),
            end_pt=(0, unit, 0),
            shaft_radius=shaft_radius,
            head_radius=head_radius,
            res=12,
            c="green",
            alpha=alpha,
        )

        z_axisArrow = Arrow(
            start_pt=(0, 0, 0),
            end_pt=(0, 0, unit),
            shaft_radius=shaft_radius,
            head_radius=head_radius,
            res=12,
            c="blue",
            alpha=alpha,
        )

        originDot = Sphere(pos=[0, 0, 0], c="black", r=0.10 * unit)

        F = x_axisArrow + y_axisArrow + z_axisArrow + originDot
        return F

    def getLocalFrameMatrix(self, R_ij, t_ij):
        """
        Homogeneous transform matrix T_ij from rotation R_ij and translation t_ij.
        """
        t_ij = np.array(t_ij, dtype=float).reshape(3, 1)
        T_ij = np.block([
            [R_ij, t_ij],
            [np.zeros((1, 3)), np.array([[1.0]])]
        ])
        return T_ij

    def forward_kinematics(self, Phi):
        """
        Returns:
            T_00, T_01, T_02, T_03, T_04, e
        where e is the end-effector position.
        """
        radius = 0.4
        phi1, phi2, phi3, phi4 = Phi

        # Base
        R_00 = self.RotationMatrix(0, axis_name="z")
        t_00 = self.arm_location.copy()
        t_00[-1, 0] = 0.0
        T_00 = self.getLocalFrameMatrix(R_00, t_00)

        # Frame 1
        R_01 = self.RotationMatrix(phi1, axis_name="z")
        t_01 = self.arm_location.copy()
        T_01 = self.getLocalFrameMatrix(R_01, t_01)

        # Frame 2
        R_12 = self.RotationMatrix(phi2, axis_name="y")
        t_12 = np.array([[0.0], [0.0], [self.L1 + 2 * radius]])
        T_12 = self.getLocalFrameMatrix(R_12, t_12)
        T_02 = T_01 @ T_12

        # Frame 3
        R_23 = self.RotationMatrix(phi3, axis_name="y")
        t_23 = np.array([[0.0], [0.0], [self.L2 + 2 * radius]])
        T_23 = self.getLocalFrameMatrix(R_23, t_23)
        T_03 = T_01 @ T_12 @ T_23

        # Frame 4 / end effector
        R_34 = self.RotationMatrix(phi4, axis_name="y")
        t_34 = np.array([[-28.4], [0.0], [self.L3 + radius]])
        T_34 = self.getLocalFrameMatrix(R_34, t_34)
        T_04 = T_01 @ T_12 @ T_23 @ T_34

        e = T_04[0:3, -1]
        return T_00, T_01, T_02, T_03, T_04, e

    def get_pose_transforms(self, Phi):
        """
        Convenience function for rendering.
        """
        T_00, T_01, T_02, T_03, T_04, _ = self.forward_kinematics(Phi)
        return [T_00, T_01, T_02, T_03, T_04]

    def update_pose(self, Phi):
        """
        Update the robot's existing meshes in-place.
        """
        transforms = self.get_pose_transforms(Phi)

        Base  = self.source_parts[0].clone()
        Part1 = self.source_parts[1].clone()
        Part2 = self.source_parts[2].clone()
        Part3 = self.source_parts[3].clone()
        Part4 = self.createCoordinateFrameMesh()

        new_meshes = [Base, Part1, Part2, Part3, Part4]

        for mesh, T in zip(new_meshes, transforms):
            mesh.apply_transform(LinearTransform(T))

        self.meshes = new_meshes
        return self.meshes

    def jacobian_matrix(self, phi):
        """
        Numerical Jacobian of end-effector position wrt all 4 joint angles.
        Output shape: (3, 4)
        """
        step = self.delta_phi
        phi = np.array(phi, dtype=float).copy()

        _, _, _, _, _, e = self.forward_kinematics(phi)

        cols = []
        for i in range(4):
            phi_step = phi.copy()
            phi_step[i] += step
            _, _, _, _, _, e_step = self.forward_kinematics(phi_step)
            de = (e_step - e) / step
            cols.append(de.reshape(3, 1))

        J = np.concatenate(cols, axis=1)
        return J

    def ik_step(self, phi, step_scale=None):
        """
        Perform ONE IK update toward self.target.
        Use this inside your simulation loop for smooth reactive control.
        """
        if step_scale is None:
            step_scale = self.target_lambda

        phi = np.array(phi, dtype=float).copy()
        target = np.array(self.target, dtype=float)

        _, _, _, _, _, e = self.forward_kinematics(phi)

        J = self.jacobian_matrix(phi)
        J_pinv = np.linalg.pinv(J)

        e_delta = step_scale * (target - e)
        phi_delta = J_pinv @ e_delta
        phi = phi + phi_delta

        _, _, _, _, _, e_new = self.forward_kinematics(phi)
        return phi, e_new

    def inverse_kinematics_newton(self, initial_phi, record_every=20, motion_threshold=10):
        """
        Runs IK and stores only joint angles.

        Returns
        -------
        trajectory : (N, 4) ndarray
            Recorded joint-angle states.
        """
        phi = np.array(initial_phi, dtype=float).copy()
        _, _, _, _, _, e = self.forward_kinematics(phi)

        target = np.array(self.target, dtype=float)
        target_lambda = self.target_lambda
        convergence = self.convergence
        target_tolerance = self.target_tolerance
        iteration_limit = self.iteration_limit

        recorder = [phi.copy()]
        iteration = 0
        e_accumulate = 0.0

        while np.linalg.norm(target - e) > target_tolerance:
            iteration += 1

            J = self.jacobian_matrix(phi)
            J_pinv = np.linalg.pinv(J)

            e_delta = target_lambda * (target - e)
            phi_delta = J_pinv @ e_delta
            phi = phi + phi_delta

            e_previous = e
            _, _, _, _, _, e = self.forward_kinematics(phi)
            e_accumulate += np.linalg.norm(e_previous - e)

            if iteration % record_every == 0 or e_accumulate > motion_threshold:
                recorder.append(phi.copy())
                e_accumulate = 0.0

            if np.linalg.norm(e_previous - e) < convergence:
                break

            if iteration > iteration_limit:
                break

        if not np.allclose(recorder[-1], phi):
            recorder.append(phi.copy())

        return np.array(recorder)

    def end_effector_xy(self, phi):
        """
        2D projection of the end-effector for use as a crowd obstacle.
        """
        _, _, _, _, _, e = self.forward_kinematics(phi)
        return e[:2]

    def set_target_from_xy(self, xy, z_height=60.0):
        """
        Helper to convert a 2D crowd target into a 3D robot target.
        """
        self.target = np.array([float(xy[0]), float(xy[1]), float(z_height)], dtype=float)

    #@title Robot helper functions for the crowd simulation

def obstacle_repulsion_from_circle(particle_xy, obstacle_center, obstacle_radius, gain=2500.0, buffer=20.0):
    """
    Repulsive force from a circular obstacle in 2D.
    """
    particle_xy = np.array(particle_xy, dtype=float)
    obstacle_center = np.array(obstacle_center, dtype=float)

    dvec = particle_xy - obstacle_center
    dist = np.linalg.norm(dvec) + 1e-9
    safe_dist = obstacle_radius + buffer

    if dist >= safe_dist:
        return np.zeros(2)

    direction = dvec / dist
    strength = gain * (1.0 / dist - 1.0 / safe_dist) / (dist * dist)
    return strength * direction


def particles_near_exits(positions, exits, exit_radius=120.0):
    """
    Returns indices of particles close to at least one exit.
    """
    positions = np.asarray(positions, dtype=float)
    exits = [np.asarray(e, dtype=float) for e in exits]

    near = []
    for i, p in enumerate(positions):
        for ex in exits:
            if np.linalg.norm(p - ex) <= exit_radius:
                near.append(i)
                break
    return np.array(near, dtype=int)


def simple_kmeans(X, k=2, max_iter=25, seed=0):
    """
    Small NumPy-only k-means to avoid sklearn dependency.
    """
    X = np.asarray(X, dtype=float)
    n = len(X)

    if n == 0:
        return None, None

    k = max(1, min(k, n))
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(n, size=k, replace=False)].copy()
    labels = np.zeros(n, dtype=int)

    for _ in range(max_iter):
        dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        new_labels = np.argmin(dists, axis=1)

        new_centers = centers.copy()
        for j in range(k):
            pts = X[new_labels == j]
            if len(pts) > 0:
                new_centers[j] = pts.mean(axis=0)

        if np.array_equal(new_labels, labels) and np.allclose(new_centers, centers):
            break

        labels = new_labels
        centers = new_centers

    return labels, centers


def detect_exit_cluster(positions, velocities, exits, exit_radius=120.0, k=2, min_cluster_size=4):
    """
    Detect the most significant cluster among particles near exits.
    Returns dict or None.
    """
    positions = np.asarray(positions, dtype=float)
    velocities = np.asarray(velocities, dtype=float)

    idx = particles_near_exits(positions, exits, exit_radius=exit_radius)
    if len(idx) < min_cluster_size:
        return None

    X = positions[idx]
    V = velocities[idx]

    labels, centers = simple_kmeans(X, k=k, max_iter=25, seed=0)
    if labels is None:
        return None

    best = None
    best_score = -np.inf

    for j in range(len(centers)):
        mask = (labels == j)
        pts = X[mask]
        vel = V[mask]

        if len(pts) < min_cluster_size:
            continue

        centroid = pts.mean(axis=0)
        mean_vel = vel.mean(axis=0)
        speed = np.linalg.norm(mean_vel)

        score = len(pts) + 2.0 * speed

        if score > best_score:
            best_score = score
            best = {
                "centroid": centroid,
                "velocity": mean_vel,
                "size": len(pts),
                "indices": idx[mask]
            }

    return best


    def predict_cluster_position(cluster, exits, horizon=10.0, exit_bias=0.15):
        """
        Predict future cluster position from centroid and mean velocity.
        """
        if cluster is None:
            return None

        centroid = np.asarray(cluster["centroid"], dtype=float)
        velocity = np.asarray(cluster["velocity"], dtype=float)

        pred = centroid + horizon * velocity

        # Slight bias toward nearest exit to stabilize prediction
        nearest_exit = min(exits, key=lambda ex: np.linalg.norm(centroid - np.asarray(ex, dtype=float)))
        nearest_exit = np.asarray(nearest_exit, dtype=float)
        pred = (1.0 - exit_bias) * pred + exit_bias * nearest_exit

        return pred

time: 7.53 ms (started: 2026-03-19 02:34:49 -04:00)


In [7]:
# -----------------------------
#@title Clear video frames
# -----------------------------
frames_dir = "frames"
os.makedirs(frames_dir, exist_ok=True)

for f in glob.glob(os.path.join(frames_dir, "*.png")):
    os.remove(f)

print("Old frames cleared.")

Old frames cleared.
time: 1.62 ms (started: 2026-03-19 02:34:49 -04:00)


In [8]:
#@title Scene setup and robot creation
#@markdown This cell loads the STL files, creates the three robots, and prepares the common scene.

import os
import glob
import numpy as np
from vedo import load, Axes, Circle, Sphere

# --------------------------------------------------
# Settings
# --------------------------------------------------
unit = 1
BaseH = 105 / unit
BaseRotH = 81 / unit
HumerusH = 217 / unit
RadiusH = 416 / unit

BaseX = 10 / unit
BaseY = 10 / unit

Circle_radius = 550 / unit

BasePos = [
    [Circle_radius, 0],
    [Circle_radius * np.cos(120 * np.pi / 180),  Circle_radius * np.sin(120 * np.pi / 180)],
    [Circle_radius * np.cos(-120 * np.pi / 180), Circle_radius * np.sin(-120 * np.pi / 180)],
]

L = [BaseRotH, HumerusH, RadiusH, 0]

view_angle = [1, 1, 1]

# --------------------------------------------------
# Locate robot folder
# --------------------------------------------------
candidate_dirs = ["robot", "robot/"]
robot_dir = None

for d in candidate_dirs:
    if os.path.isdir(d):
        robot_dir = d
        break

if robot_dir is None:
    raise FileNotFoundError(
        "Could not find the robot folder. Make sure robot.zip was downloaded and extracted."
    )

print("Using robot directory:", robot_dir)

# --------------------------------------------------
# List STL files for debugging
# --------------------------------------------------
stl_files = sorted(glob.glob(os.path.join(robot_dir, "*.stl")))
print("STL files found:")
for f in stl_files:
    print(" -", os.path.basename(f))

if not stl_files:
    raise FileNotFoundError(f"No STL files found inside {robot_dir}")

# --------------------------------------------------
# Safe STL loader
# --------------------------------------------------
def load_part(filename, color_name):
    path = os.path.join(robot_dir, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing STL file: {path}")

    mesh = load(path)
    if mesh is None:
        raise ValueError(f"vedo.load() failed for: {path}")

    return mesh.color(color_name)

# --------------------------------------------------
# Load meshes
# --------------------------------------------------
Base    = load_part("Base.stl", "blue5")
BaseRot = load_part("BaseRot.stl", "lightblue")
Humerus = load_part("Humerus.stl", "gray5")
Radius  = load_part("Radius.stl", "red5")

parts = [Base, BaseRot, Humerus, Radius]

# --------------------------------------------------
# Arm locations
# --------------------------------------------------
arm_location0 = np.array([[BasePos[0][0]], [BasePos[0][1]], [BaseH]], dtype=float)
arm_location1 = np.array([[BasePos[1][0]], [BasePos[1][1]], [BaseH]], dtype=float)
arm_location2 = np.array([[BasePos[2][0]], [BasePos[2][1]], [BaseH]], dtype=float)

# --------------------------------------------------
# Create robot instances
# --------------------------------------------------
myRobot0 = RobotArm(L, parts, arm_location0)
myRobot1 = RobotArm(L, parts, arm_location1)
myRobot2 = RobotArm(L, parts, arm_location2)

# --------------------------------------------------
# Scene actors
# --------------------------------------------------
axes = Axes(
    xrange=(-Circle_radius * 1.2, Circle_radius * 1.2),
    yrange=(-Circle_radius * 1.2, Circle_radius * 1.2),
    zrange=(0, Circle_radius * 1.2),
)

circle = Circle(pos=(0, 0, 0), r=Circle_radius, res=120, c="lightgray", alpha=1.0)
ball = Sphere(myRobot0.target, r=30).c("red")

print("Scene setup complete.")

Using robot directory: robot
STL files found:
 - Base.stl
 - BaseRot.stl
 - Humerus.stl
 - Radius.stl
Scene setup complete.
time: 820 ms (started: 2026-03-19 02:34:49 -04:00)


In [9]:
#@title Crowd helper functions

import numpy as np

def obstacle_repulsion_from_circle(particle_xy, obstacle_center, obstacle_radius, gain=2500.0, buffer=20.0):
    particle_xy = np.array(particle_xy, dtype=float)
    obstacle_center = np.array(obstacle_center, dtype=float)

    dvec = particle_xy - obstacle_center
    dist = np.linalg.norm(dvec) + 1e-9
    safe_dist = obstacle_radius + buffer

    if dist >= safe_dist:
        return np.zeros(2)

    direction = dvec / dist
    strength = gain * (1.0 / dist - 1.0 / safe_dist) / (dist * dist)
    return strength * direction


def particles_near_exits(positions, exits, exit_radius=120.0):
    positions = np.asarray(positions, dtype=float)
    exits = [np.asarray(e, dtype=float) for e in exits]

    near = []
    for i, p in enumerate(positions):
        for ex in exits:
            if np.linalg.norm(p - ex) <= exit_radius:
                near.append(i)
                break
    return np.array(near, dtype=int)


def simple_kmeans(X, k=2, max_iter=25, seed=0):
    X = np.asarray(X, dtype=float)
    n = len(X)

    if n == 0:
        return None, None

    k = max(1, min(k, n))
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(n, size=k, replace=False)].copy()
    labels = np.zeros(n, dtype=int)

    for _ in range(max_iter):
        dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        new_labels = np.argmin(dists, axis=1)

        new_centers = centers.copy()
        for j in range(k):
            pts = X[new_labels == j]
            if len(pts) > 0:
                new_centers[j] = pts.mean(axis=0)

        if np.array_equal(new_labels, labels) and np.allclose(new_centers, centers):
            break

        labels = new_labels
        centers = new_centers

    return labels, centers


def detect_exit_cluster(positions, velocities, exits, exit_radius=120.0, k=2, min_cluster_size=4):
    positions = np.asarray(positions, dtype=float)
    velocities = np.asarray(velocities, dtype=float)

    idx = particles_near_exits(positions, exits, exit_radius=exit_radius)
    if len(idx) < min_cluster_size:
        return None

    X = positions[idx]
    V = velocities[idx]

    labels, centers = simple_kmeans(X, k=k, max_iter=25, seed=0)
    if labels is None:
        return None

    best = None
    best_score = -np.inf

    for j in range(len(centers)):
        mask = (labels == j)
        pts = X[mask]
        vel = V[mask]

        if len(pts) < min_cluster_size:
            continue

        centroid = pts.mean(axis=0)
        mean_vel = vel.mean(axis=0)
        speed = np.linalg.norm(mean_vel)

        score = len(pts) + 2.0 * speed

        if score > best_score:
            best_score = score
            best = {
                "centroid": centroid,
                "velocity": mean_vel,
                "size": len(pts),
                "indices": idx[mask]
            }

    return best


def predict_cluster_position(cluster, exits, horizon=10.0, exit_bias=0.15):
    if cluster is None:
        return None

    centroid = np.asarray(cluster["centroid"], dtype=float)
    velocity = np.asarray(cluster["velocity"], dtype=float)

    pred = centroid + horizon * velocity

    nearest_exit = min(exits, key=lambda ex: np.linalg.norm(centroid - np.asarray(ex, dtype=float)))
    nearest_exit = np.asarray(nearest_exit, dtype=float)
    pred = (1.0 - exit_bias) * pred + exit_bias * nearest_exit

    return pred

time: 2.92 ms (started: 2026-03-19 02:34:50 -04:00)


In [10]:
#@title Simple crowd setup

import numpy as np

np.random.seed(7)

# 2D room size
ROOM_W = 500.0
ROOM_H = 300.0

# exits at bottom edge
exits = [
    np.array([120.0, 0.0]),
    np.array([380.0, 0.0]),
]

# particle initialization
N = 80
positions = np.column_stack([
    np.random.uniform(80, 420, N),
    np.random.uniform(180, 280, N),
]).astype(float)

velocities = np.zeros((N, 2), dtype=float)

# simulation constants
dt = 0.05
num_steps = 250
particle_speed = 18.0
robot_radius = 35.0

def nearest_exit(p):
    return min(exits, key=lambda ex: np.linalg.norm(p - ex))

def update_crowd(positions, velocities, robot_xy=None):
    positions = positions.copy()
    velocities = velocities.copy()

    for i in range(len(positions)):
        p = positions[i]
        ex = nearest_exit(p)

        desired_dir = ex - p
        desired_norm = np.linalg.norm(desired_dir) + 1e-9
        desired_dir = desired_dir / desired_norm

        desired_vel = particle_speed * desired_dir

        # smooth toward exit
        velocities[i] = 0.85 * velocities[i] + 0.15 * desired_vel

        # mild particle repulsion
        for j in range(len(positions)):
            if i == j:
                continue
            dvec = positions[i] - positions[j]
            dist = np.linalg.norm(dvec) + 1e-9
            if dist < 12.0:
                velocities[i] += 4.0 * (dvec / dist)

        # robot repulsion
        if robot_xy is not None:
            velocities[i] += dt * obstacle_repulsion_from_circle(
                particle_xy=positions[i],
                obstacle_center=robot_xy,
                obstacle_radius=robot_radius,
                gain=2500.0,
                buffer=20.0
            )

        # update position
        positions[i] += dt * velocities[i]

        # keep particles inside room except exits
        positions[i, 0] = np.clip(positions[i, 0], 0, ROOM_W)
        positions[i, 1] = np.clip(positions[i, 1], 0, ROOM_H)

    return positions, velocities

time: 12.3 ms (started: 2026-03-19 02:34:50 -04:00)


In [11]:
#@title Integrated robot + crowd simulation (clean 2D blocking visualization)

import matplotlib.pyplot as plt
import os
import glob
import numpy as np

frames_dir = "crowd_robot_frames"
os.makedirs(frames_dir, exist_ok=True)

for f in glob.glob(os.path.join(frames_dir, "*.png")):
    os.remove(f)

# Visible 2D "end-effector obstacle" location in the room
robot_xy = np.array([250.0, 220.0], dtype=float)

cluster_history = []

for t in range(num_steps):
    # -------------------------------------------------
    # 1) update crowd with current robot obstacle
    # -------------------------------------------------
    robot_xy_now = robot_xy.copy()
    positions, velocities = update_crowd(positions, velocities, robot_xy=robot_xy_now)

    # -------------------------------------------------
    # 2) detect escaping cluster near exits
    # -------------------------------------------------
    cluster = detect_exit_cluster(
        positions=positions,
        velocities=velocities,
        exits=exits,
        exit_radius=120.0,
        k=2,
        min_cluster_size=4
    )

    pred_xy = None

    # -------------------------------------------------
    # 3) move visible blocking obstacle toward prediction
    # -------------------------------------------------
    if cluster is not None:
        pred_xy = predict_cluster_position(
            cluster,
            exits=exits,
            horizon=10.0,
            exit_bias=0.15
        )

        # smooth motion toward predicted location
        alpha = 0.15
        robot_xy = (1.0 - alpha) * robot_xy + alpha * pred_xy
        cluster_history.append((cluster["centroid"].copy(), pred_xy.copy()))

    # keep obstacle inside room bounds
    robot_xy[0] = np.clip(robot_xy[0], 0, ROOM_W)
    robot_xy[1] = np.clip(robot_xy[1], 0, ROOM_H)

    # -------------------------------------------------
    # 4) draw 2D frame for report/video
    # -------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))

    # particles
    ax.scatter(positions[:, 0], positions[:, 1], s=18, label="particles")

    # exits
    for ex in exits:
        ax.scatter(ex[0], ex[1], marker="s", s=120, label="exit")

    # robot obstacle
    circ = plt.Circle((robot_xy[0], robot_xy[1]), robot_radius, fill=False, color="red", linewidth=2)
    ax.add_patch(circ)
    ax.scatter(robot_xy[0], robot_xy[1], color="red", marker="o", s=80, label="end-effector")

    # cluster + prediction
    if cluster is not None:
        centroid = cluster["centroid"]
        ax.scatter(centroid[0], centroid[1], marker='x', s=120, label='cluster')
        ax.scatter(pred_xy[0], pred_xy[1], marker='*', s=160, label='predicted')

    ax.set_xlim(0, ROOM_W)
    ax.set_ylim(0, ROOM_H)
    ax.set_title(f"Robot Interference Simulation - step {t}")
    ax.set_aspect("equal")

    handles, labels = ax.get_legend_handles_labels()
    seen = set()
    unique_handles = []
    unique_labels = []
    for h, l in zip(handles, labels):
        if l not in seen:
            unique_handles.append(h)
            unique_labels.append(l)
            seen.add(l)
    ax.legend(unique_handles, unique_labels, loc="upper right")

    plt.tight_layout()
    plt.savefig(os.path.join(frames_dir, f"frame_{t:03d}.png"), dpi=120)
    plt.close(fig)

print("Simulation finished.")
print(f"Saved frames to: {frames_dir}")

Simulation finished.
Saved frames to: crowd_robot_frames
time: 37.4 s (started: 2026-03-19 02:34:50 -04:00)


In [12]:
#@title Create crowd+robot video (Windows-safe, no resizing warning)

import os
import sys
import glob
import subprocess

for pkg in ["imageio", "pillow", "imageio-ffmpeg"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import imageio.v2 as imageio

frames = sorted(glob.glob(os.path.join("crowd_robot_frames", "frame_*.png")))

if not frames:
    raise FileNotFoundError("No PNG frames found in crowd_robot_frames/")

video_path = "crowd_robot.mp4"

with imageio.get_writer(video_path, fps=20, macro_block_size=1) as writer:
    for f in frames:
        img = imageio.imread(f)
        writer.append_data(img)

print(f"Video created: {video_path}")
print(f"Number of frames: {len(frames)}")

Video created: crowd_robot.mp4
Number of frames: 250
time: 5.23 s (started: 2026-03-19 02:35:27 -04:00)


In [13]:
#@title Display crowd+robot video

import os
from IPython.display import HTML
from base64 import b64encode

video_path = "crowd_robot.mp4"

if not os.path.exists(video_path):
    raise FileNotFoundError(f"{video_path} was not created yet.")

with open(video_path, "rb") as f:
    mp4 = f.read()

decoded_vid = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width="900" controls>
      <source src="{decoded_vid}" type="video/mp4">
</video>
""")

time: 5.44 ms (started: 2026-03-19 02:35:33 -04:00)


In [14]:
#@title Render animation frames from stored joint angles (3 robots)

#@markdown Reconstructs robot poses from stored histories: phi_history0, phi_history1, phi_history2

import os
import glob
import numpy as np

required = ["phi_history0", "phi_history1", "phi_history2"]
for name in required:
    if name not in globals() or len(globals()[name]) == 0:
        raise ValueError(f"{name} is empty or not defined.")

frames_dir = "robot_frames"
os.makedirs(frames_dir, exist_ok=True)

for f in glob.glob(os.path.join(frames_dir, "*.png")):
    os.remove(f)

idxlimit = max(len(phi_history0), len(phi_history1), len(phi_history2))

for idx in range(idxlimit):
    phi0 = np.array(phi_history0[min(idx, len(phi_history0) - 1)], dtype=float)
    phi1 = np.array(phi_history1[min(idx, len(phi_history1) - 1)], dtype=float)
    phi2 = np.array(phi_history2[min(idx, len(phi_history2) - 1)], dtype=float)

    meshes0 = myRobot0.update_pose(phi0)
    meshes1 = myRobot1.update_pose(phi1)
    meshes2 = myRobot2.update_pose(phi2)

    plt_vtk.clear()
    plt_vtk += axes
    plt_vtk += circle
    plt_vtk += ball

    for actor in meshes0:
        plt_vtk += actor
    for actor in meshes1:
        plt_vtk += actor
    for actor in meshes2:
        plt_vtk += actor

    plt_vtk.render()
    plt_vtk.screenshot(os.path.join(frames_dir, f"frame_{idx:03d}.png"))

print(f"Saved {idxlimit} robot render frames to: {frames_dir}")

ValueError: phi_history0 is empty or not defined.

time: 260 ms (started: 2026-03-19 02:35:33 -04:00)


In [ ]:
#@title Show animation sequence as a grid

import glob
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# SETTINGS
# -----------------------------
GRID_ROWS = 4
GRID_COLS = 6

# -----------------------------
# Load frames
# -----------------------------
frame_files = sorted(glob.glob(f"{frames_dir}/frame_*.png"))
N = len(frame_files)

grid_size = GRID_ROWS * GRID_COLS

if N == 0:
    raise RuntimeError("No frames found.")

# -----------------------------
# Compute sampling indices
# -----------------------------
if N <= grid_size:
    indices = np.arange(N)
else:
    step = N / grid_size
    indices = (np.arange(grid_size) * step).astype(int)

# -----------------------------
# Display grid preview
# -----------------------------
print("Displaying frame grid preview (sampled across sequence)...")

fig, axes = plt.subplots(GRID_ROWS, GRID_COLS, figsize=(12, 8))

for i in range(grid_size):
    ax = axes.flat[i]
    ax.axis("off")

    if i < len(indices):
        img = Image.open(frame_files[indices[i]])
        ax.imshow(img)

plt.tight_layout()
plt.show()

In [ ]:
#@title Create video from frames
# -----------------------------
# Create video
# -----------------------------
!ffmpeg -loglevel error -y -framerate 30 -i frames/frame_%03d.png \
        -c:v libx264 -pix_fmt yuv420p -movflags +faststart spheres.mp4

print("Video created: spheres.mp4")

In [ ]:
#@title Display video in notebook


from IPython.display import HTML
from base64 import b64encode

video_path = 'spheres.mp4'
# Read the file and encode it
mp4 = open(video_path, 'rb').read()
decoded_vid = "data:video/mp4;base64," + b64encode(mp4).decode()

# Embed in an HTML video tag
HTML(f'<video width=500 controls><source src={decoded_vid} type="video/mp4"></video>')